In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

db_path = Path("../data/olist.db")
conn = sqlite3.connect(db_path)


def get_df(filename):
    path = Path(f"../sql/{filename}")
    text = path.read_text()
    df = pd.read_sql(text,conn)
    return df



In [2]:
print(pd.read_sql("SELECT SUM(items_value) FROM orders WHERE in_revenue = 1", conn))

get_df("01_revenue_by_category.sql")["revenue"].sum() 

get_df("01_revenue_by_category.sql")[get_df("01_revenue_by_category.sql")["category"] == "unknown"]


   SUM(items_value)
0       13221498.11


,category,revenue,n_of_orders,n_of_items,percentage_of_total
20,unknown,170726.63,1392,1537,1.29


In [3]:
get_df("01_revenue_by_category.sql").head(2)

,category,revenue,n_of_orders,n_of_items,percentage_of_total
0,health_beauty,1233131.72,8647,9465,9.33
1,watches_gifts,1166176.98,5495,5859,8.82


sql/01_revenue_by_category.sql calculates the revenue of each product category, counting only delivered orders (in_revenue = 1). Revenue comes from order_items.price (item level), not from the order totals, because joining the orders table to the items would repeat each order once per item.

The query returns 74 categories, including unknown (products with no category, which still have sales). The sum of the category revenues is 13221498.11, identical to the total revenue in the orders table, so no rows were lost or duplicated.

-The top category is health_beauty, with 1,233,131.72 (9.33% of the revenue), followed by watches_gifts (8.82%) and bed_bath_table (7.74%).

-The top 5 categories account for about 39.8% of the revenue, so the revenue is spread across many categories instead of depending on a few.

-health_beauty is first in revenue but not in number of orders: bed_bath_table has more (9,272 vs. 8,647), which suggests that health_beauty has a higher average price per order.

In [4]:
df_02 = get_df("02_monthly_orders_revenue.sql")
df_02

,month_of_purchase,n_of_orders,revenue,average_ticket,revenue_variation
0,2017-01,750,111798.36,149.06,NaN
1,2017-02,1653,234223.40,141.70,109.51
2,2017-03,2546,359198.85,141.08,53.36
3,2017-04,2303,340669.68,147.92,-5.16
4,2017-05,3546,489338.25,138.00,43.64
5,2017-06,3135,421923.37,134.58,-13.78
6,2017-07,3872,481604.52,124.38,14.15
7,2017-08,4193,554699.70,132.29,15.18
8,2017-09,4150,607399.67,146.36,9.50
9,2017-10,4478,648247.65,144.76,6.73


In [5]:
max_revenue_month = df_02.loc[[df_02["revenue"].idxmax()], "month_of_purchase"]
max_n_of_orders = df_02.loc[[df_02["n_of_orders"].idxmax()], "month_of_purchase"]
pd.DataFrame({"best_month_revenue": max_revenue_month, "best_month_n_of_orders": max_n_of_orders})


,best_month_revenue,best_month_n_of_orders
10,2017-11,2017-11


In [22]:
print(f"total orders: {df_02['n_of_orders'].sum()}")
print(f"total revenue: {round(df_02['revenue'].sum(),2)}")
pd.read_sql("SELECT SUM(items_value) FROM orders WHERE in_revenue = 1 AND month_of_purchase >= '2017-01'",conn)


total orders: 96211
total revenue: 13181027.13


,SUM(items_value)
0,13181027.13


In [23]:
first_semestre_2017 = df_02[df_02["month_of_purchase"] < "2017-07"]["revenue"].sum()
first_semestre_2018 = df_02[(df_02["month_of_purchase"] > "2017-12") & (df_02["month_of_purchase"] < "2018-07")]["revenue"].sum()
print(f"Revenue from 1º semestre 2017 to 1º semestre 2018: {round((first_semestre_2018 - first_semestre_2017)/(first_semestre_2017)*100, 2)}")

print(first_semestre_2018, first_semestre_2017)

Revenue from 1º semestre 2017 to 1º semestre 2018: 181.61
5511595.0200000005 1957151.9100000001


sql/02_monthly_orders_revenue.sql returns, for each month, the number of orders, the revenue, the average ticket and the revenue variation compared with the previous month. Only delivered orders are counted (in_revenue = 1), and the period is 2017-01 to 2018-08.

I excluded 2016 because it has only three months (September, October and December), with 267 orders and one missing month (November). A month-over-month comparison with such a small base gives meaningless values (for example, +29,777%).

-The peak is November 2017, with 7,289 orders and 987,765 in revenue. It is a possible Black Friday effect, but the data does not confirm it.

-In the first half of 2018 the revenue was about 2.8 times the first half of 2017 (+181.61%), but from 2018 the monthly revenue stays between about 826 thousand and 978 thousand, so the growth stops.

-The average ticket changes little (between 124 and 149), so the revenue growth comes from the number of orders and not from higher prices.

-The sum of the monthly revenue (13,181,027.13) and of the orders (96,211) matches the totals of the orders table for the same period.